# AGN-Egent — Tutorial

Agentic spectral decomposition of AGN: separate the **broad** and **narrow** emission-line
components of a quasar/AGN spectrum, with an LLM-driven QC layer that verifies each step.

This notebook walks through the pipeline on the bundled example SDSS quasar.


## 0. Setup

Pin BLAS threads **before** importing numpy — PyQSOFit's fit is only reproducible single-threaded.
(`import config; config.pin_threads()` does this; `import agn_egent` also pins as a fallback.)


In [ ]:
import config; config.pin_threads()
import agn_egent
from agn_egent import (load_sdss, decompose, QsoparConfig, verify,
                       run_agent, RuleInspector, ClaudeInspector, derive,
                       render_diagnostic)


## 1. Load a spectrum

`load_sdss` reads a standard SDSS spec FITS into a `Spectrum` (observed-frame wavelength, flux, error, redshift).


In [ ]:
spec = load_sdss('external/PyQSOFit/example/data/spec-0332-52367-0639.fits')
spec


## 2. One-shot decomposition + verification

`decompose()` fits power-law + Fe II + host galaxy + tied narrow lines + broad Gaussians and
returns faithfully separated component arrays. `verify()` runs deterministic QC checks.


In [ ]:
result = decompose(spec, make_figure=False)
print(result.summary())
report = verify(result)
print(report.summary())


## 3. The agentic QC loop

`run_agent` drives decompose -> verify -> render -> inspect -> remedy -> repeat. The default
`RuleInspector` is deterministic and needs no API key; swap in `ClaudeInspector()` (with
`ANTHROPIC_API_KEY` set) to put a vision LLM in the seat. `finalize_mc=True` adds 1-sigma errors.


In [ ]:
outcome = run_agent(spec, inspector=RuleInspector(), finalize_mc=True, nsamp=25)
print(outcome.summary())


On this object the agent flags the degenerate 2-Gaussian broad H-beta, applies `set_ngauss=1`,
and then accepts — the remaining warnings are intrinsic data limits, not fixable by a model edit.


## 4. Derived quantities (black-hole mass)

`derive()` turns the broad H-beta FWHM + continuum luminosity into a single-epoch M_BH
(Vestergaard & Peterson 2006), L_bol, and Eddington ratio, propagating the MC uncertainties.


In [ ]:
dq = derive(outcome.final_result, 'Hb')
print(dq)


## 5. Diagnostic figure

The QC figure (components + a pull panel) is what the LLM reviewer sees.


In [ ]:
from IPython.display import Image
path = render_diagnostic(outcome.final_result, 'data/runs/tutorial/diagnostic.png')
Image(filename=path)


## 6. Batch mode

`run_batch` decomposes a list of spectra in parallel (one process each); the inspector is wrapped
in `TriageInspector` so the LLM only reviews the WARN/FAIL objects. Results come back as a table.


In [ ]:
from agn_egent import run_batch
data = 'external/PyQSOFit/example/data/'
report = run_batch([data+'spec-0266-51602-0013.fits', data+'spec-0332-52367-0639.fits'],
                   max_workers=2)
print(report.summary())
report.to_csv('data/runs/tutorial/results.csv')


## Next steps

- Put a real LLM in the seat: `run_agent(spec, inspector=ClaudeInspector(), ...)`
- Command line: `python run_agn.py 388-51793-445 --inspector claude`
- Accuracy benchmark vs. Shen DR7: `python tests/phase6_literature.py`
